# 02 — LUNA16 radiomic feature extraction

Run pyradiomics on every 3D patch with its sphere ROI mask. Outputs `results/luna16_cache/features.npz` with `X`, `feature_names`, `y`. Failed extractions are dropped before the matrix is saved.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import yaml

from utils.seed import set_seed
from utils.features_radiomics import build_feature_matrix

with open('../configs/luna16.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
patches = np.load(cache_dir / 'patches_3d.npy')
masks = np.load(cache_dir / 'masks_3d.npy')
labels_df = pd.read_csv(cache_dir / 'labels.csv')
print('patches', patches.shape, 'masks', masks.shape, 'labels', len(labels_df))

patches (1070, 64, 64, 64) masks (1070, 64, 64, 64) labels 1070


In [3]:
X, feature_names = build_feature_matrix(patches, masks, verbose=True)
print('X shape:', X.shape, '# features:', len(feature_names))

y = labels_df['label'].to_numpy().astype(int)

valid = ~np.any(~np.isfinite(X), axis=1) if X.size else np.zeros(0, dtype=bool)
print(f'valid rows: {valid.sum()}/{len(valid)}')

X = X[valid]
y = y[valid]
meta = labels_df.iloc[valid].reset_index(drop=True)

pyradiomics:   0%|          | 0/1070 [00:00<?, ?it/s]

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
pyradiomics:   0%|          | 1/1070 [00:00<06:57,  2.56it/s]GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
pyradiomics:   0%|          | 4/1070 [00:00<01:56,  9.12it/s]GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
pyradiomics:   1%|          | 7/1070 [00:00<01:18, 13.54it/s]GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, o

X shape: (1070, 91) # features: 91
valid rows: 1070/1070


In [4]:
out = cache_dir / 'features.npz'
np.savez_compressed(
    out,
    X=X,
    y=y,
    feature_names=np.array(feature_names),
    seriesuid=meta['seriesuid'].to_numpy(),
)
print('saved →', out)

saved → ..\..\results\luna16_cache\features.npz
